# AI-Powered Sustainable Agriculture Optimization System

## What this notebook does:

This notebook builds an intelligent system that:

* Identifies inefficient resource usage (water, fertilizer)  
* Measures sustainability using real-world efficiency metrics  
* Incorporates market insights for better decision-making  
* Generates actionable recommendations for farmers  

## Key Highlights:

- Dual-dataset integration (Farm + Market intelligence)
- Advanced feature engineering (efficiency + waste metrics)
- Classification of sustainability levels
- Explainable AI (SHAP)
- Decision system for real-world impact

## Load Data

In [ ]:
import pandas as pd

# Load datasets
farmer_df = pd.read_csv("/content/drive/MyDrive/Agri_DS/farmer_advisor_dataset.csv")
market_df = pd.read_csv("/content/drive/MyDrive/Agri_DS/market_researcher_dataset.csv")

print("Farmer Data Shape:", farmer_df.shape)
print("Market Data Shape:", market_df.shape)


In [ ]:
print("Farmer Columns:\n", farmer_df.columns)
print("\nMarket Columns:\n", market_df.columns)

## Dataset Overview

We are working with two datasets:

### 1. Farmer Advisor Dataset
Contains farm-level data:
- Resource usage (water, fertilizer, etc.)
- Environmental conditions
- Crop yield

### 2. Market Research Dataset
Contains:
- Market demand
- Price trends
- External agricultural insights

## Strategy

We will:
- Analyze both datasets
- Use farmer data for sustainability modeling
- Use market data for decision enhancement

In [ ]:
farmer_df.info()

In [ ]:
market_df.info()

In [ ]:
farmer_df.describe()

## Data Understanding

### Farmer Dataset Features

| Feature | Description |
|--------|------------|
| Soil_pH | Soil acidity level |
| Soil_Moisture | Water content in soil |
| Temperature_C | Temperature |
| Rainfall_mm | Rainfall |
| Fertilizer_Usage_kg | Fertilizer input |
| Pesticide_Usage_kg | Chemical usage |
| Crop_Yield_ton | Output |
| Sustainability_Score | Predefined sustainability metric |

---

### Market Dataset Features

| Feature | Description |
|--------|------------|
| Market_Price_per_ton | Selling price |
| Demand_Index | Market demand |
| Supply_Index | Availability |
| Consumer_Trend_Index | Market trend |

## EDA

In [ ]:
# Select only numerical columns
numeric_df = farmer_df.select_dtypes(include=['float64', 'int64'])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix (Numerical Features Only)")
plt.show()

In [ ]:
# Resource vs Yield
sns.scatterplot(x='Fertilizer_Usage_kg', y='Crop_Yield_ton', data=farmer_df)
plt.title("Fertilizer vs Yield")
plt.show()

sns.scatterplot(x='Pesticide_Usage_kg', y='Crop_Yield_ton', data=farmer_df)
plt.title("Pesticide vs Yield")
plt.show()

## EDA Insights

- Increasing fertilizer does not always increase yield  
- Pesticide overuse shows diminishing returns  
- Sustainability score varies widely → strong modeling opportunity  

## Feature Engineering for Sustainability

We create meaningful real-world metrics:

1. Fertilizer Efficiency → Yield per fertilizer  
2. Pesticide Efficiency → Yield per pesticide  
3. Resource Intensity → Total input usage  
4. Climate Stress → Temperature + Rainfall deviation  

In [ ]:
df = farmer_df.copy()

# Efficiency features
df['fertilizer_efficiency'] = df['Crop_Yield_ton'] / (df['Fertilizer_Usage_kg'] + 1)
df['pesticide_efficiency'] = df['Crop_Yield_ton'] / (df['Pesticide_Usage_kg'] + 1)

# Resource intensity
df['resource_intensity'] = df['Fertilizer_Usage_kg'] + df['Pesticide_Usage_kg']

# Climate stress
df['climate_stress'] = abs(df['Temperature_C'] - 25) + abs(df['Rainfall_mm'] - 175)

In [ ]:
def classify(score):
    if score > 70:
        return "High"
    elif score > 40:
        return "Medium"
    else:
        return "Low"

df['sustainability_class'] = df['Sustainability_Score'].apply(classify)

In [ ]:
# Random merge simulation (real-world assumption)
market_sample = market_df.sample(n=len(df), random_state=42).reset_index(drop=True)

df = pd.concat([df.reset_index(drop=True), market_sample], axis=1)

## Why This Matters

We now combine:
- Farm efficiency
- Market demand
- Price signals

**This mimics real-world agricultural decision systems**

## MODEL BUILDING

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['target'] = le.fit_transform(df['sustainability_class'])

X = df.select_dtypes(include=['float64', 'int64']).drop(['target'], axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)

## EVALUATION

In [ ]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    preds = model.predict(X_test)
    print(name, "Accuracy:", accuracy_score(y_test, preds))

## Explainable AI

In [ ]:
import shap

model = models["RandomForest"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

## SHAP Insights

- Fertilizer efficiency strongly improves sustainability  
- High resource intensity reduces sustainability  
- Market demand influences decision-making  

## Decision System

In [ ]:
def recommendation(row):
    rec = []

    if row['fertilizer_efficiency'] < 0.05:
        rec.append("Reduce fertilizer usage")

    if row['pesticide_efficiency'] < 0.1:
        rec.append("Optimize pesticide usage")

    if row['resource_intensity'] > 150:
        rec.append("High resource consumption detected")

    if row['Demand_Index'] < 50:
        rec.append("Low market demand – consider crop switch")

    if row['Market_Price_per_ton'] < row['Competitor_Price_per_ton']:
        rec.append("Pricing disadvantage detected")

    return rec

In [ ]:
df['recommendations'] = df.apply(recommendation, axis=1)

df[['sustainability_class', 'recommendations']].head()

## UI

In [ ]:
# ============================================
# 🌱 Sustainable Agriculture Optimization AI
# FINAL VERSION (Production-Ready)
# ============================================

import pandas as pd
import numpy as np
import gradio as gr
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# -------------------------------
# 📌 Load Data
# -------------------------------
farmer_df = pd.read_csv("/content/drive/MyDrive/Agri_DS/farmer_advisor_dataset.csv")
market_df = pd.read_csv("/content/drive/MyDrive/Agri_DS/market_researcher_dataset.csv")

df = farmer_df.copy()

# -------------------------------
# 🔥 Feature Engineering
# -------------------------------
df['fertilizer_efficiency'] = df['Crop_Yield_ton'] / (df['Fertilizer_Usage_kg'] + 1)
df['pesticide_efficiency'] = df['Crop_Yield_ton'] / (df['Pesticide_Usage_kg'] + 1)
df['resource_intensity'] = df['Fertilizer_Usage_kg'] + df['Pesticide_Usage_kg']
df['climate_stress'] = abs(df['Temperature_C'] - 25) + abs(df['Rainfall_mm'] - 175)

# -------------------------------
# 🎯 Target Creation
# -------------------------------
def classify(score):
    if score > 70:
        return "High"
    elif score > 40:
        return "Medium"
    else:
        return "Low"

df['sustainability_class'] = df['Sustainability_Score'].apply(classify)

# -------------------------------
# 📊 Merge Market Data
# -------------------------------
market_sample = market_df.sample(n=len(df), random_state=42).reset_index(drop=True)
df = pd.concat([df.reset_index(drop=True), market_sample], axis=1)

# -------------------------------
# 🧠 Feature Selection (FIXED)
# -------------------------------
feature_cols = [
    'Soil_pH',
    'Soil_Moisture',
    'Temperature_C',
    'Rainfall_mm',
    'Fertilizer_Usage_kg',
    'Pesticide_Usage_kg',
    'Crop_Yield_ton',
    'fertilizer_efficiency',
    'pesticide_efficiency',
    'resource_intensity',
    'climate_stress',
    'Market_Price_per_ton',
    'Demand_Index',
    'Supply_Index',
    'Competitor_Price_per_ton',
    'Economic_Indicator',
    'Weather_Impact_Score',
    'Consumer_Trend_Index'
]

# -------------------------------
# 🔤 Encode Target
# -------------------------------
le = LabelEncoder()
df['target'] = le.fit_transform(df['sustainability_class'])

X = df[feature_cols]
y = df['target']

# -------------------------------
# 🤖 Train Model
# -------------------------------
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# -------------------------------
# 🚨 Prediction Function
# -------------------------------
def predict_sustainability(
    soil_ph,
    soil_moisture,
    temperature,
    rainfall,
    fertilizer,
    pesticide,
    yield_ton,
    market_price,
    demand,
    supply,
    competitor_price,
    economic_indicator,
    weather_score,
    consumer_trend
):
    try:
        # Feature Engineering
        fertilizer_eff = yield_ton / (fertilizer + 1)
        pesticide_eff = yield_ton / (pesticide + 1)
        resource_intensity = fertilizer + pesticide
        climate_stress = abs(temperature - 25) + abs(rainfall - 175)

        # Input Data
        input_data = pd.DataFrame([{
            'Soil_pH': soil_ph,
            'Soil_Moisture': soil_moisture,
            'Temperature_C': temperature,
            'Rainfall_mm': rainfall,
            'Fertilizer_Usage_kg': fertilizer,
            'Pesticide_Usage_kg': pesticide,
            'Crop_Yield_ton': yield_ton,
            'fertilizer_efficiency': fertilizer_eff,
            'pesticide_efficiency': pesticide_eff,
            'resource_intensity': resource_intensity,
            'climate_stress': climate_stress,
            'Market_Price_per_ton': market_price,
            'Demand_Index': demand,
            'Supply_Index': supply,
            'Competitor_Price_per_ton': competitor_price,
            'Economic_Indicator': economic_indicator,
            'Weather_Impact_Score': weather_score,
            'Consumer_Trend_Index': consumer_trend
        }])

        # Align columns
        input_data = input_data[feature_cols]

        # Prediction
        pred = model.predict(input_data)[0]
        label = le.inverse_transform([pred])[0]

        # -------------------------------
        # 📌 Recommendation System
        # -------------------------------
        recommendations = []

        if fertilizer_eff < 0.05:
            recommendations.append("🌿 Reduce fertilizer usage")

        if pesticide_eff < 0.1:
            recommendations.append("🐛 Optimize pesticide usage")

        if resource_intensity > 150:
            recommendations.append("⚠️ High resource consumption detected")

        if demand < 50:
            recommendations.append("📉 Low demand – consider crop change")

        if market_price < competitor_price:
            recommendations.append("💰 Improve pricing strategy")

        if not recommendations:
            recommendations.append("✅ Sustainable farming practice")

        return label, "\n".join(recommendations)

    except Exception as e:
        return "Error", str(e)

# -------------------------------
# 🎨 Professional UI Layout
# -------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🌱 Sustainable Agriculture Optimization AI")
    gr.Markdown("### AI system for resource efficiency & smart farming decisions")

    with gr.Row():
        with gr.Column():
            soil_ph = gr.Slider(5.5, 7.5, label="Soil pH")
            soil_moisture = gr.Slider(10, 50, label="Soil Moisture")
            temperature = gr.Slider(15, 35, label="Temperature (°C)")
            rainfall = gr.Slider(50, 300, label="Rainfall (mm)")
            fertilizer = gr.Slider(50, 200, label="Fertilizer Usage (kg)")
            pesticide = gr.Slider(1, 20, label="Pesticide Usage (kg)")
            yield_ton = gr.Slider(1, 10, label="Crop Yield (ton)")

        with gr.Column():
            market_price = gr.Slider(100, 1000, label="Market Price")
            demand = gr.Slider(0, 100, label="Demand Index")
            supply = gr.Slider(0, 100, label="Supply Index")
            competitor_price = gr.Slider(100, 1000, label="Competitor Price")
            economic_indicator = gr.Slider(0, 100, label="Economic Indicator")
            weather_score = gr.Slider(0, 100, label="Weather Impact")
            consumer_trend = gr.Slider(0, 100, label="Consumer Trend")

    submit_btn = gr.Button("🚀 Analyze Sustainability")

    output_label = gr.Textbox(label="🌍 Sustainability Level")
    output_rec = gr.Textbox(label="📌 Recommendations")

    submit_btn.click(
        predict_sustainability,
        inputs=[
            soil_ph, soil_moisture, temperature, rainfall,
            fertilizer, pesticide, yield_ton,
            market_price, demand, supply, competitor_price,
            economic_indicator, weather_score, consumer_trend
        ],
        outputs=[output_label, output_rec]
    )

# -------------------------------
# 🚀 Launch App
# -------------------------------
app.launch()

## Final Insights

- Sustainability depends on efficiency, not resource usage  
- Overuse of fertilizers and pesticides reduces long-term productivity  
- Market-aware farming improves profitability  

## Business Impact

- Reduce farmer costs  
- Promote sustainable agriculture  
- Enable AI-driven decisions  

---

## What Makes This Project Strong

* Real-world feature engineering  
* Explainable AI  
* Actionable recommendations  
* Business + technical thinking  